In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import joblib

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

In [3]:
df = pd.read_csv('synthetic_soil_bearing_capacity.csv')

In [4]:
df.head()

,soil_type,moisture_content,bulk_density,grain_size,foundation_depth,field_observation_score,bearing_capacity,safety_class
0,Sand,28.35,16.97,Fine,1.72,3,237.45,Safe
1,Sand,28.42,14.61,Coarse,1.98,4,231.42,Safe
2,Gravel,6.18,20.01,Coarse,2.05,3,400.00,Safe
3,Silt,9.35,14.20,Fine,1.27,4,131.96,Moderate
4,Sand,30.75,19.53,Medium,1.60,5,235.48,Safe


In [5]:
df['soil_type'].unique()

array(['Sand', 'Gravel', 'Silt', 'Clay'], dtype=object)

In [6]:
df['moisture_content'].describe()

count    1800.000000
mean       22.525744
std        10.056967
min         5.000000
25%        14.050000
50%        22.690000
75%        31.202500
max        39.960000
Name: moisture_content, dtype: float64

In [7]:
df['grain_size'].unique()

array(['Fine', 'Coarse', 'Medium'], dtype=object)

In [8]:
df['safety_class'].unique()

array(['Safe', 'Moderate', 'Unsafe'], dtype=object)

In [9]:
df['foundation_depth'].describe()

count    1800.000000
mean        1.763311
std         0.722040
min         0.500000
25%         1.160000
50%         1.780000
75%         2.400000
max         3.000000
Name: foundation_depth, dtype: float64

In [10]:
df['bearing_capacity'].describe()

count    1800.000000
mean      211.660583
std        81.711552
min        50.000000
25%       151.890000
50%       210.835000
75%       267.922500
max       400.000000
Name: bearing_capacity, dtype: float64

In [11]:
print(df.groupby('safety_class')['bearing_capacity'].max())
print(df.groupby('safety_class')['bearing_capacity'].min())

safety_class
Moderate    199.94
Safe        400.00
Unsafe       99.63
Name: bearing_capacity, dtype: float64
safety_class
Moderate    100.10
Safe        200.03
Unsafe       50.00
Name: bearing_capacity, dtype: float64


In [12]:
X = df.drop(columns=['bearing_capacity', 'safety_class'])
y = df['bearing_capacity']

In [13]:
X.shape, y.shape

((1800, 6), (1800,))

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1440, 6), (360, 6), (1440,), (360,))

In [16]:
transformer = ColumnTransformer(
    transformers = [
        ('tnf', OneHotEncoder(sparse_output=False, drop='first'), ['soil_type', 'grain_size'])
    ],
    remainder='passthrough'
)

In [23]:
from lightgbm import LGBMRegressor

pipeline = Pipeline([
    ('transformer', transformer),
    ('regressor', LGBMRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ))
])

In [24]:
pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000264 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 761
[LightGBM] [Info] Number of data points in the train set: 1440, number of used features: 9
[LightGBM] [Info] Start training from score 212.466729
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,steps,"[('transformer', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tnf', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
joblib.dump(pipeline, "model1.pkl")

['model1.pkl']

In [ ]:
# X_train_new = transformer.fit_transform(X_train)
# X_test_new = transformer.transform(X_test)

In [ ]:
# lr = LinearRegression()

In [ ]:
# lr.fit(X_train_new, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [ ]:
# pred = lr.predict(X_test_new)

In [ ]:
# r2_score(y_test, pred)

0.914155227089074

In [ ]:
# np.sqrt(mean_squared_error(y_test, pred))

np.float64(24.177073839019204)

In [ ]:
# model = LinearRegression().fit(X_train_new, y_train)

In [ ]:
# joblib.dump(model, "model.pkl")

['model.pkl']